# M2M PLIMS Data Access Testing

#### Intro
Starting with the first deployment of the Coastal Pioneer Mid-Atlantic Bight (MAB) Array, OOI collected data from a plankton imaging instrument (McLane Research Labs Imaging Flow Cytobot) deployed at 7 meters depth on the near-surface instrument frame. The introduction of a new instrument to the physical OOI infrastructure required new parsing code and metadata to ingest the data into the OOI Cyberinfrastructure. Once ingested, the data will be available for machine-to-machine (M2M) download from OOI's THREDDS server and then pushed to Data Explorer on a near-real-time basis.

#### Purpose
Telemetered data from the first PLIMS deployment on Central Surface Mooring has been ingested into the Dev0 development platform where the relevant parsers are being tested. After the parameters and relevant metadata were decided upon, test NetCDFs were created for review to confirm that the parsers could create a valid and accurate dataset. This data pathway from ingested data to datasets asynchronously compiled on demand now must be tested from the perspective of a data user.

#### Objectives:

* Test Output: Request HDR science and engineering datasets with different request parameters.
* Dataset Review: Look for incorrect data values, types or metadata to provide feedback to CI. 

#### Supporting tools
If you want to run this notebook as-is, you will need to clone A. Reed's [ooinet repo](https://github.com/reedan88/OOINet) to your local machine and install it as a local dev repo (which adds it to your base path). You'll also need the [ooi_data_explorations repo](https://github.com/oceanobservatories/ooi-data-explorations). Lastly, you'll want to install the ```ioos_qc``` [python package](https://github.com/ioos/ioos_qc). 

Whether individual sections of this notebook can successfully request, download, and analyze data depends on which data is ingested in the Dev0 environment.


In [1]:
# Import libraries
import os
import re
import gc
import io
import ast
import netrc
import pandas as pd
import numpy as np
import xarray as xr
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")
import glob

# Import progress bar display from dask
from dask.diagnostics import ProgressBar

# Import pyplot and show plots inline
import matplotlib.pyplot as plt
%matplotlib inline 

# Import OOI-developed modules
from ooinet import M2M
from ooinet.Instrument.common import process_file
from ooi_data_explorations.common import get_vocabulary, load_gc_thredds, gc_collect

---
## Request and load the data

Search the production server for available datasets

In [2]:
M2M.URLS

{'data': 'https://ooinet.oceanobservatories.org/api/m2m/12576/sensor/inv',
 'anno': 'https://ooinet.oceanobservatories.org/api/m2m/12580/anno/find',
 'vocab': 'https://ooinet.oceanobservatories.org/api/m2m/12586/vocab/inv',
 'asset': 'https://ooinet.oceanobservatories.org/api/m2m/12587',
 'deploy': 'https://ooinet.oceanobservatories.org/api/m2m/12587/events/deployment/inv',
 'preload': 'https://ooinet.oceanobservatories.org/api/m2m/12575/parameter',
 'cal': 'https://ooinet.oceanobservatories.org/api/m2m/12587/asset/cal',
 'fileServer': 'https://opendap.oceanobservatories.org/thredds/fileServer/',
 'dodsC': 'https://opendap.oceanobservatories.org/thredds/dodsC/',
 'goldCopy': 'https://thredds.dataexplorer.oceanobservatories.org/thredds/catalog/ooigoldcopy/public/',
 'goldCopy_fileServer': 'https://thredds.dataexplorer.oceanobservatories.org/thredds/fileServer/',
 'goldCopy_dodsC': 'https://thredds.dataexplorer.oceanobservatories.org/thredds/dodsC/'}

In [3]:
ds_df = M2M.search_datasets(array="CP10CNSM")
ds_df.reset_index(inplace=True)
# ds_df

Searching https://ooinet.oceanobservatories.org/api/m2m/12576/sensor/inv/CP10CNSM


Find the available datastreams for a given **refdes**

In [ ]:
# url = ds_df["url"][0]
# url = "/".join((url, 'telemetered', 'plims_a_hdr_instrument'))
# # Query the preload data
# preload_data = M2M.get_api(url)
# preload_data

In [2]:
datastreams = M2M.get_datastreams("CP10CNSM-RID27-07-PLIMSA000")
datastreams

,refdes,method,stream
0,CP10CNSM-RID27-07-PLIMSA000,recovered_host,plims_a_adc_instrument
1,CP10CNSM-RID27-07-PLIMSA000,recovered_host,plims_a_hdr_engineering
2,CP10CNSM-RID27-07-PLIMSA000,recovered_host,plims_a_hdr_instrument
3,CP10CNSM-RID27-07-PLIMSA000,recovered_inst,plims_a_adc_instrument
4,CP10CNSM-RID27-07-PLIMSA000,recovered_inst,plims_a_hdr_engineering
5,CP10CNSM-RID27-07-PLIMSA000,recovered_inst,plims_a_hdr_instrument
6,CP10CNSM-RID27-07-PLIMSA000,telemetered,plims_a_adc_instrument
7,CP10CNSM-RID27-07-PLIMSA000,telemetered,plims_a_hdr_engineering
8,CP10CNSM-RID27-07-PLIMSA000,telemetered,plims_a_hdr_instrument


In [7]:
# Define a generic preprocessing routine. Do NOT use any of the ooi_data_explorations "process_instrument" methods. We want to be comparing "apples-to-apples" 
def preprocess(ds):
    ds = process_file(ds)
    return ds

#### Telemetered HDR engineering data stream

In [3]:
# Setup parameters needed to request data
refdes = datastreams.refdes[7]
method = datastreams.method[7]
stream = datastreams.stream[7]

In [4]:
# Import vocabulary
site, node, sensor = refdes.split('-', 2)
vocab = get_vocabulary(site, node, sensor)

In [5]:
# Use the gold copy THREDDs datasets
thredds_url = M2M.get_thredds_url(refdes, method, stream, beginDT="2024-07-20T00:00:00.0", endDT="2024-08-01T00:00:00.0")
# Get the THREDDs catalog
thredds_catalog = M2M.get_thredds_catalog(thredds_url)
thredds_catalog = [x for x in thredds_catalog if "blank" not in x]

# Clean the THREDDs catalog
sensor_files, ancillary_files = M2M.clean_catalog(thredds_catalog, stream)

# Generate the urls to access and load the data
sensor_files = [re.sub("catalog.html\?dataset=", M2M.URLS["dodsC"], file) for file in sensor_files]



Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process


In [8]:
# Load the data
with ProgressBar():
    data_hdreng = xr.open_mfdataset(sensor_files, parallel=True, preprocess=preprocess)
    # Uncomment the following lines to load data
    # without preprocessing and swap dims to time:
    # data_hdreng = xr.open_mfdataset(sensor_files, parallel=True)
    # data_hdreng = data_hdreng.swap_dims({"obs":"time"})

[########################################] | 100% Completed | 1.83 ss


In [9]:
# Check dataset content
print(data_hdreng.source)
data_hdreng

CP10CNSM-RID27-07-PLIMSA000-telemetered-plims_a_hdr_engineering


<xarray.Dataset> Size: 16kB
Dimensions:               (time: 62)
Coordinates:
  * time                  (time) datetime64[ns] 496B 2024-07-20T00:24:54 ... ...
Data variables: (12/45)
    pump1_state           (time) float32 248B dask.array<chunksize=(62,), meta=np.ndarray>
    refill_debubble       (time) float32 248B dask.array<chunksize=(62,), meta=np.ndarray>
    prime_sample          (time) float32 248B dask.array<chunksize=(62,), meta=np.ndarray>
    deployment            (time) int32 248B dask.array<chunksize=(62,), meta=np.ndarray>
    alt_interval          (time) float64 496B dask.array<chunksize=(62,), meta=np.ndarray>
    internal_temperature  (time) float64 496B dask.array<chunksize=(62,), meta=np.ndarray>
    ...                    ...
    alt_pmta_threshold    (time) float32 248B dask.array<chunksize=(62,), meta=np.ndarray>
    focus_motor_step_lg   (time) float64 496B dask.array<chunksize=(62,), meta=np.ndarray>
    syringes_autorun      (time) float64 496B dask.array<chunksize=(62,), meta=np.ndarray>
    auto_start            (time) float32 248B dask.array<chunksize=(62,), meta=np.ndarray>
    detergent_rinse_vol   (time) float32 248B dask.array<chunksize=(62,), meta=np.ndarray>
    sample_volume_skip    (time) float64 496B dask.array<chunksize=(62,), meta=np.ndarray>
Attributes: (12/69)
    node:                               RID27
    comment:                            Data collected from the OOI M2M API a...
    publisher_email:                    
    sourceUrl:                          http://oceanobservatories.org/
    collection_method:                  telemetered
    stream:                             plims_a_hdr_engineering
    ...                                 ...
    geospatial_lon_resolution:          0.1
    geospatial_vertical_units:          meters
    geospatial_vertical_resolution:     0.1
    geospatial_vertical_positive:       down
    lat:                                35.94988
    lon:                                -75.11943

In [ ]:
# plot any data
data_hdreng.roi_count.plot()

In [ ]:
# For logical values, check that the mean
# of values matches the expected result
# - 1 for True
# - 0 for False
data_hdreng.beads_stirrer.mean().values

In [ ]:
# Check dtype for internal temperature
# This requires precision of float64 to match raw data.
print(f"Internal temperature dtype: {data_hdreng.internal_temperature.dtype}")

In [ ]:
# Check that the time variable is coming from
# internal timestamp and the raw data filename
# This does not work if the preprocess routine is 
# used to  remove the preferred timestamp variable
# data_hdreng["preferred_timestamp"][0].values

In [ ]:
# Check for standard names left in the dataset
names = list(())
for var in data_hdreng.variables:
    try:
        names.append(data_hdreng[var].standard_name)
    except:
        continue
print(f"Remaining standard names: {names}")

In [ ]:
# telem_hdreng_path = os.path.abspath(os.path.join("./plims_data", f"dev0_telem_hdr_eng_{datetime.now().isoformat(timespec='seconds')}.nc"))
# telem_hdreng_path = os.path.abspath(os.path.join("./plims_data", f"dev0_telem_hdr_eng.nc"))
# telem_hdreng_path = re.sub(":", "", telem_hdreng_path)
# data_hdreng.to_netcdf(telem_hdreng_path, engine="netcdf4")

#### Telemetered HDR science data stream

In [ ]:
# Setup parameters needed to request data
refdes = datastreams.refdes[8]
method = datastreams.method[8]
stream = datastreams.stream[8]

In [ ]:
# Import vocabulary
site, node, sensor = refdes.split('-', 2)
vocab = get_vocabulary(site, node, sensor)

In [ ]:
# Use the gold copy THREDDs datasets
thredds_url = M2M.get_thredds_url(refdes, method, stream, beginDT="2024-05-01T00:00:00.0", endDT="2024-08-01T00:00:00.0")
# Get the THREDDs catalog
thredds_catalog = M2M.get_thredds_catalog(thredds_url)
thredds_catalog = [x for x in thredds_catalog if "blank" not in x]

# Clean the THREDDs catalog
sensor_files, ancillary_files = M2M.clean_catalog(thredds_catalog, stream)

# Generate the urls to access and load the data
sensor_files = [re.sub("catalog.html\?dataset=", M2M.URLS["dodsC"], file) for file in sensor_files]

In [ ]:
# Load the data
with ProgressBar():
    data_hdrsci = xr.open_mfdataset(sensor_files, parallel=True, preprocess=preprocess)

In [ ]:
# Check dataset content
data_hdrsci

In [ ]:
# Check for standard names left in the dataset
names = list(())
for var in data_hdrsci.variables:
    try:
        names.append(data_hdrsci[var].standard_name)
    except:
        continue
print(f"Remaining standard names: {names}")

In [ ]:
# Check updated parameter attributes
data_hdrsci.blob_gap_min.comment

In [ ]:
# Check dtype for the following parameters
# These require precision of float64 to match raw data.
print(f"Internal humidity dtype: {data_hdrsci.internal_humidity.dtype}")
print(f"Run time dtype: {data_hdrsci.run_time.dtype}")
print(f"Inhibit time dtype: {data_hdrsci.inhibit_time.dtype}")

In [ ]:
# For logical values, check that the mean
# of values matches the expected result
# - 1 for True
# - 0 for False
data_hdrsci.run_fast.mean().values

In [ ]:
data_hdrsci.roi_count.plot(x="time")

In [ ]:
data_hdrsci.volume_analyzed.plot.scatter(x="time", edgecolors="none", s=16)

In [ ]:
data_hdrsci.rois_per_ml.plot(x="time")

In [ ]:
t_inhibit = data_hdrsci.inhibit_time.compute()
t_run = data_hdrsci.run_time.compute()
print(t_inhibit.head())
print(t_run.head())

In [ ]:
calculated_look_time = t_run - t_inhibit
calculated_look_time.values == data_hdrsci.look_time.values

In [ ]:
fig, ax = plt.subplots(2,1, sharex=True, layout="compact")
data_hdrsci.look_time.plot(ax=ax[0])
ax[0].grid()
calculated_look_time.plot(ax=ax[1])
ax[1].grid()
ax[1].set_ylabel("Computed Look Time [s]")

In [ ]:
calculated_volume_analyzed = (t_run - t_inhibit)/(data_hdrsci.sample_volume - data_hdrsci.sample_speed)

In [ ]:
# data_hdrsci.to_netcdf(f"./plims_data/dev0_telemetered_hdr_sci_{datetime.now()}.nc")

#### Telemetered ADC science data stream

In [10]:
# Setup parameters needed to request data
refdes = datastreams.refdes[6]
method = datastreams.method[6]
stream = datastreams.stream[6]

In [11]:
# Import vocabulary
site, node, sensor = refdes.split('-', 2)
vocab = get_vocabulary(site, node, sensor)

In [12]:
# Use the gold copy THREDDs datasets
thredds_url = M2M.get_thredds_url(refdes, method, stream, beginDT="2024-05-01T00:00:00.0", endDT="2024-05-08T00:00:00.0")
# Get the THREDDs catalog
thredds_catalog = M2M.get_thredds_catalog(thredds_url)
thredds_catalog = [x for x in thredds_catalog if "blank" not in x]

# Clean the THREDDs catalog
sensor_files, ancillary_files = M2M.clean_catalog(thredds_catalog, stream)

# Generate the urls to access and load the data
sensor_files = [re.sub("catalog.html\?dataset=", M2M.URLS["dodsC"], file) for file in sensor_files]

Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process


In [13]:
# Load the data
with ProgressBar():
    data_adcsci = xr.open_mfdataset(sensor_files, parallel=True, preprocess=preprocess)

[########################################] | 100% Completed | 2.74 ss


In [14]:
# Check dataset content
data_adcsci

<xarray.Dataset> Size: 11MB
Dimensions:                     (time: 80054)
Coordinates:
  * time                        (time) datetime64[ns] 640kB 2024-05-01T00:25:...
Data variables: (12/21)
    inhibit_time_tr             (time) float32 320kB dask.array<chunksize=(80054,), meta=np.ndarray>
    peak_b                      (time) float32 320kB dask.array<chunksize=(80054,), meta=np.ndarray>
    peak_a                      (time) float32 320kB dask.array<chunksize=(80054,), meta=np.ndarray>
    grab_time_start             (time) float32 320kB dask.array<chunksize=(80054,), meta=np.ndarray>
    roi_status                  (time) float64 640kB dask.array<chunksize=(80054,), meta=np.ndarray>
    roi_width                   (time) float64 640kB dask.array<chunksize=(80054,), meta=np.ndarray>
    ...                          ...
    roi_y                       (time) float64 640kB dask.array<chunksize=(80054,), meta=np.ndarray>
    start_byte                  (time) float64 640kB dask.array<chunksize=(80054,), meta=np.ndarray>
    run_time_tr                 (time) float32 320kB dask.array<chunksize=(80054,), meta=np.ndarray>
    roi_height                  (time) float64 640kB dask.array<chunksize=(80054,), meta=np.ndarray>
    deployment                  (time) int32 320kB dask.array<chunksize=(80054,), meta=np.ndarray>
    sample_filename             (time) object 640kB dask.array<chunksize=(80054,), meta=np.ndarray>
Attributes: (12/69)
    node:                               RID27
    comment:                            Data collected from the OOI M2M API a...
    publisher_email:                    
    sourceUrl:                          http://oceanobservatories.org/
    collection_method:                  telemetered
    stream:                             plims_a_adc_instrument
    ...                                 ...
    geospatial_lon_resolution:          0.1
    geospatial_vertical_units:          meters
    geospatial_vertical_resolution:     0.1
    geospatial_vertical_positive:       down
    lat:                                35.94988
    lon:                                -75.11943

In [ ]:
# Check size of the dataset
data_adcsci.nbytes

In [ ]:
# Check preferred timestamp
# Preferred timestamp not found in variables
# if the preprocess routine is applied
# data_adcsci.preferred_timestamp[0].values

In [ ]:
# Check time values for triggers compared to sample time
print(data_adcsci.sample_timestamp[0].values) # Sample start time
print(data_adcsci.grab_time_start[0].values) # Time elapsed in seconds since the sample start time
print(data_adcsci.time[0].values) # Time of the first trigger of the sample

In [ ]:
# Calculate trigger times with grab_time_start + sample_timestamp 
grab_time_s =  data_adcsci["grab_time_start"].to_numpy()*1000000000
grab_time_s = grab_time_s.astype("timedelta64[ns]")
t_trigger_computed = grab_time_s+data_adcsci["sample_timestamp"]

In [ ]:
# Plot peak PMT A with OOINet-provided time 
# and our computed trigger time paramters
peak_a = data_adcsci.peak_a
fig = plt.figure(figsize=(8,4))
peak_a.plot.scatter(x="time", s=16, edgecolors="none", label="OOINet-provided time")
plt.scatter(t_trigger_computed, peak_a.values, s=6, edgecolors="none", label="Computed trigger time")
plt.grid()
plt.legend()

In [ ]:
# Check for parameters removed from an earlier version of the ADC data stream code
parameters = ["inhibit_time", "run_time", "look_time"]
for p in parameters:
    try:
        p_is_here = data_adcsci.p
    except:
        print(f"{p} is not found in ADC science dataset")
    else:
        print(f"{p} is still in ADC science dataset")

In [ ]:
# Check for standard names left in the dataset
names = list(())
for var in data_adcsci.variables:
    try:
        names.append(data_adcsci[var].standard_name)
    except:
        continue
print(f"Remaining standard names: {names}")

In [ ]:
data_adcsci.roi_y[0:5000].plot.scatter(x="time", edgecolors="none", s=16)

In [ ]:
# data_adcsci.to_netcdf(f"./plims_data/dev0_telemetered_adc_sci_{datetime.now()}.nc")

#### Recovered host HDR engineering data stream

In [ ]:
# Setup parameters needed to request data
refdes = datastreams.refdes[1]
method = datastreams.method[1]
stream = datastreams.stream[1]

In [ ]:
# Import vocabulary
site, node, sensor = refdes.split('-', 2)
vocab = get_vocabulary(site, node, sensor)

In [ ]:
# Use the gold copy THREDDs datasets
thredds_url = M2M.get_thredds_url(refdes, method, stream, beginDT="2024-05-01T00:00:00.0", endDT="2024-08-01T00:00:00.0")
# Get the THREDDs catalog
thredds_catalog = M2M.get_thredds_catalog(thredds_url)
thredds_catalog = [x for x in thredds_catalog if "blank" not in x]

# Clean the THREDDs catalog
sensor_files, ancillary_files = M2M.clean_catalog(thredds_catalog, stream)

# Generate the urls to access and load the data
sensor_files = [re.sub("catalog.html\?dataset=", M2M.URLS["dodsC"], file) for file in sensor_files]



In [ ]:
# Load the data
with ProgressBar():
    data_hdreng2 = xr.open_mfdataset(sensor_files, parallel=True, preprocess=preprocess)

In [ ]:
# Check dataset content
data_hdreng2

In [ ]:
# Check for standard names left in the dataset
names = list(())
for var in data_hdreng2.variables:
    try:
        names.append(data_hdreng2[var].standard_name)
    except:
        continue
print(f"Remaining standard names: {names}")

In [ ]:
# For logical values, check that the mean
# of values matches the expected result
# - 1 for True
# - 0 for False
data_hdreng2.pump1_state.mean().values

In [ ]:
# Check dtype for internal temperature
# This requires precision of float64 to match raw data.
print(f"Internal temperature dtype: {data_hdreng2.internal_temperature.dtype}")

In [ ]:
# Check that the time variable is coming from
# internal timestamp and the raw data filename.
# Preferred timestamp not found in variables if
# the preprocess routine is applied.
# data_hdreng2["preferred_timestamp"][0].values

In [ ]:
data_hdreng2.alt_sample_counter

In [ ]:
# data_hdreng2.to_netcdf("./plims_data/dev0_recoveredhost_hdr_eng.nc")

#### Recovered host HDR science data stream

In [ ]:
# Setup parameters needed to request data
refdes = datastreams.refdes[2]
method = datastreams.method[2]
stream = datastreams.stream[2]

In [ ]:
# Import vocabulary
site, node, sensor = refdes.split('-', 2)
vocab = get_vocabulary(site, node, sensor)

In [ ]:
# Use the gold copy THREDDs datasets
thredds_url = M2M.get_thredds_url(refdes, method, stream, beginDT="2024-05-01T00:00:00.0", endDT="2024-08-01T00:00:00.0")
# Get the THREDDs catalog
thredds_catalog = M2M.get_thredds_catalog(thredds_url)
thredds_catalog = [x for x in thredds_catalog if "blank" not in x]

# Clean the THREDDs catalog
sensor_files, ancillary_files = M2M.clean_catalog(thredds_catalog, stream)

# Generate the urls to access and load the data
sensor_files = [re.sub("catalog.html\?dataset=", M2M.URLS["dodsC"], file) for file in sensor_files]

In [ ]:
# Load the data
with ProgressBar():
    data_hdrsci2 = xr.open_mfdataset(sensor_files, parallel=True, preprocess=preprocess)

In [ ]:
# Check dataset content
data_hdrsci2

In [ ]:
# Check for standard names left in the dataset
names = list(())
for var in data_hdrsci2.variables:
    try:
        names.append(data_hdrsci2[var].standard_name)
    except:
        continue
print(f"Remaining standard names: {names}")

In [ ]:
# Check updated parameter attributes
data_hdrsci2.blob_gap_min.comment

In [ ]:
# Check dtype for the following parameters
# These require precision of float64 to match raw data.
print(f"Internal humidity dtype: {data_hdrsci2.internal_humidity.dtype}")
print(f"Run time dtype: {data_hdrsci2.run_time.dtype}")
print(f"Inhibit time dtype: {data_hdrsci2.inhibit_time.dtype}")

In [ ]:
# For logical values, check that the mean
# of values matches the expected result
# - 1 for True
# - 0 for False
data_hdrsci2.run_fast.mean().values

In [ ]:
# Check attributes of parameters updated since 21 Mar
data_hdrsci2.pmta_threshold

In [ ]:
# data_hdrsci.to_netcdf(f"./plims_data/dev0_recoveredhost_hdr_sci_{datetime.now()}.nc")

In [ ]:
# data_hdrsci.to_netcdf(f"./plims_data/dev0_recoveredhost_hdr_sci_{datetime.now()}.nc")

In [ ]:
# data_hdrsci.to_netcdf(f"./plims_data/dev0_recoveredhost_hdr_sci_{datetime.now()}.nc")

#### Recovered host ADC science data stream

In [ ]:
# Setup parameters needed to request data
refdes = datastreams.refdes[0]
method = datastreams.method[0]
stream = datastreams.stream[0]

In [ ]:
# Import vocabulary
site, node, sensor = refdes.split('-', 2)
vocab = get_vocabulary(site, node, sensor)

In [ ]:
# Use the gold copy THREDDs datasets
thredds_url = M2M.get_thredds_url(refdes, method, stream, beginDT="2024-05-01T00:00:00.0", endDT="2024-05-15T00:00:00.0")
# Get the THREDDs catalog
thredds_catalog = M2M.get_thredds_catalog(thredds_url)
thredds_catalog = [x for x in thredds_catalog if "blank" not in x]

# Clean the THREDDs catalog
sensor_files, ancillary_files = M2M.clean_catalog(thredds_catalog, stream)

# Generate the urls to access and load the data
sensor_files = [re.sub("catalog.html\?dataset=", M2M.URLS["dodsC"], file) for file in sensor_files]

In [ ]:
# Load the data
with ProgressBar():
    data_adcsci2 = xr.open_mfdataset(sensor_files, parallel=True, preprocess=preprocess)

In [ ]:
# Check dataset content
data_adcsci2

In [ ]:
# Check attributes of paramaters updated since 21 Mar
data_adcsci2.roi_y

In [ ]:
# Check for parameters removed from an earlier version of the ADC data stream code
parameters = ["inhibit_time", "run_time", "look_time"]
for p in parameters:
    try:
        p_is_here = data_adcsci2.p
    except:
        print(f"{p} is not found in ADC science dataset")
    else:
        print(f"{p} is still in ADC science dataset")

In [ ]:
# Check for standard names left in the dataset
names = list(())
for var in data_adcsci2.variables:
    try:
        names.append(data_adcsci2[var].standard_name)
    except:
        continue
print(f"Remaining standard names: {names}")

In [ ]:
# data_adcsci.to_netcdf(f"./plims_data/dev0_recoveredhost_adc_sci_{datetime.now()}.nc")

#### Recovered instrument ADC science data stream

In [8]:
# Setup parameters needed to request data
refdes = datastreams.refdes[3]
method = datastreams.method[3]
stream = datastreams.stream[3]

In [9]:
# Import vocabulary
site, node, sensor = refdes.split('-', 2)
vocab = get_vocabulary(site, node, sensor)

In [12]:
# Use the gold copy THREDDs datasets
thredds_url = M2M.get_thredds_url(refdes, method, stream, beginDT="2024-09-10T00:00:00.0", endDT="2024-09-20T14:00:00.0")
# Get the THREDDs catalog
thredds_catalog = M2M.get_thredds_catalog(thredds_url)
thredds_catalog = [x for x in thredds_catalog if "blank" not in x]

# Clean the THREDDs catalog
sensor_files, ancillary_files = M2M.clean_catalog(thredds_catalog, stream)

# Generate the urls to access and load the data
sensor_files = [re.sub("catalog.html\?dataset=", M2M.URLS["dodsC"], file) for file in sensor_files]

Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process
Waiting for request to process


In [13]:
# Load the data
with ProgressBar():
    # data_adcsci3 = xr.open_mfdataset(sensor_files, parallel=True, preprocess=preprocess)
    data_adcsci3 = xr.open_mfdataset(sensor_files, parallel=True)

[########################################] | 100% Completed | 2.67 ss


In [14]:
# Check dataset content
data_adcsci3

<xarray.Dataset> Size: 13MB
Dimensions:                     (obs: 43065)
Coordinates:
  * obs                         (obs) int32 172kB 0 1 2 3 ... 43062 43063 43064
    time                        (obs) datetime64[ns] 345kB dask.array<chunksize=(43065,), meta=np.ndarray>
Data variables: (12/26)
    driver_timestamp            (obs) datetime64[ns] 345kB dask.array<chunksize=(43065,), meta=np.ndarray>
    id                          (obs) |S64 3MB dask.array<chunksize=(43065,), meta=np.ndarray>
    inhibit_time_tr             (obs) float32 172kB dask.array<chunksize=(43065,), meta=np.ndarray>
    peak_b                      (obs) float32 172kB dask.array<chunksize=(43065,), meta=np.ndarray>
    peak_a                      (obs) float32 172kB dask.array<chunksize=(43065,), meta=np.ndarray>
    grab_time_start             (obs) float32 172kB dask.array<chunksize=(43065,), meta=np.ndarray>
    ...                          ...
    run_time_tr                 (obs) float32 172kB dask.array<chunksize=(43065,), meta=np.ndarray>
    roi_height                  (obs) float64 345kB dask.array<chunksize=(43065,), meta=np.ndarray>
    port_timestamp              (obs) datetime64[ns] 345kB dask.array<chunksize=(43065,), meta=np.ndarray>
    deployment                  (obs) int32 172kB dask.array<chunksize=(43065,), meta=np.ndarray>
    preferred_timestamp         (obs) object 345kB dask.array<chunksize=(43065,), meta=np.ndarray>
    sample_filename             (obs) object 345kB dask.array<chunksize=(43065,), meta=np.ndarray>
Attributes: (12/73)
    node:                               RID27
    comment:                            
    publisher_email:                    
    sourceUrl:                          http://oceanobservatories.org/
    collection_method:                  recovered_inst
    stream:                             plims_a_adc_instrument
    ...                                 ...
    geospatial_vertical_positive:       down
    lat:                                35.94988
    lon:                                -75.11943
    DODS.strlen:                        24
    DODS.dimName:                       string24
    DODS_EXTRA.Unlimited_Dimension:     obs

In [16]:
# Check format of sample_filename parameter
data_adcsci3["sample_filename"] = data_adcsci3.sample_filename.astype(str)
data_adcsci3["sample_filename"].head().compute()

<xarray.DataArray 'sample_filename' (obs: 5)> Size: 480B
array(['D20240910T002445_IFCB199', 'D20240910T002445_IFCB199',
       'D20240910T002445_IFCB199', 'D20240910T002445_IFCB199',
       'D20240910T002445_IFCB199'], dtype='<U24')
Coordinates:
  * obs      (obs) int32 20B 0 1 2 3 4
    time     (obs) datetime64[ns] 40B 2024-09-10T00:24:57.241979392 ... 2024-...
Attributes:
    comment:      File name without extension as a globally unique identifier...
    long_name:    Sample File Name
    _ChunkSizes:  [10000    24]

In [17]:
# Check shape of provenance array
data_adcsci3["provenance"].head().compute()

<xarray.DataArray 'provenance' (obs: 5)> Size: 320B
array([b'85dc670f-bf7c-4eb0-94aa-cfa0ac114b96',
       b'85dc670f-bf7c-4eb0-94aa-cfa0ac114b96',
       b'85dc670f-bf7c-4eb0-94aa-cfa0ac114b96',
       b'85dc670f-bf7c-4eb0-94aa-cfa0ac114b96',
       b'85dc670f-bf7c-4eb0-94aa-cfa0ac114b96'], dtype='|S64')
Coordinates:
  * obs      (obs) int32 20B 0 1 2 3 4
    time     (obs) datetime64[ns] 40B 2024-09-10T00:24:57.241979392 ... 2024-...
Attributes:
    name:         provenance
    _ChunkSizes:  [10000    36]

In [21]:
# Check attributes of paramaters updated since 21 Mar
data_adcsci3.roi_y

<xarray.DataArray 'roi_y' (time: 43065)> Size: 345kB
dask.array<getitem, shape=(43065,), dtype=float64, chunksize=(43065,), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 345kB 2024-09-10T00:24:57.241979392 ... 20...
Attributes:
    long_name:    ROI y-position
    comment:      Y-position of the upper left corner of the bounding box for...
    units:        pixels
    _ChunkSizes:  10000

In [22]:
# Check for parameters removed from an earlier version of the ADC data stream code
parameters = ["inhibit_time", "run_time", "look_time"]
for p in parameters:
    try:
        p_is_here = data_adcsci3.p
    except:
        print(f"{p} is not found in ADC science dataset")
    else:
        print(f"{p} is still in ADC science dataset")

inhibit_time is not found in ADC science dataset
run_time is not found in ADC science dataset
look_time is not found in ADC science dataset


In [23]:
# Check for standard names left in the dataset
names = list(())
for var in data_adcsci3.variables:
    try:
        names.append(data_adcsci2[var].standard_name)
    except:
        continue
print(f"Remaining standard names: {names}")

Remaining standard names: []


In [ ]:
# data_adcsci.to_netcdf(f"./plims_data/dev0_recoveredhost_adc_sci_{datetime.now()}.nc")

#### Check Gold Copy of ADC data

In [10]:
adc_gc = load_gc_thredds(site, node, sensor, method, stream)
# adc_gc = gc_collect(adc_id)

Merging the data files into a single dataset


In [11]:
# Preview dataset content
adc_gc

<xarray.Dataset> Size: 733MB
Dimensions:                     (time: 3186577, string36: 36, string18: 18,
                                 string24: 24)
Coordinates:
    obs                         (time) int32 13MB 0 2 3 ... 3279794 3279797
  * time                        (time) datetime64[ns] 25MB 2024-04-03T14:16:1...
Dimensions without coordinates: string36, string18, string24
Data variables: (12/26)
    driver_timestamp            (time) datetime64[ns] 25MB 2025-10-01T14:59:5...
    id                          (time, string36) |S1 115MB b'5' b'c' ... b'5'
    inhibit_time_tr             (time) float32 13MB 0.09436 0.1872 ... 101.5
    peak_b                      (time) float32 13MB 0.0403 0.01936 ... 0.06607
    peak_a                      (time) float32 13MB 3.467 0.6372 ... 0.07058
    grab_time_start             (time) float32 13MB 1.611 1.792 ... 1.201e+03
    ...                          ...
    run_time_tr                 (time) float32 13MB 1.633 1.822 ... 1.201e+03
    roi_height                  (time) uint32 13MB 154 42 98 66 ... 194 178 146
    port_timestamp              (time) datetime64[ns] 25MB 1900-01-01 ... 190...
    deployment                  (time) int32 13MB 1 1 1 1 1 1 1 ... 1 1 1 1 1 1
    preferred_timestamp         (time, string18) |S1 57MB b'i' b'n' ... b'p'
    sample_filename             (time, string24) |S1 76MB b'D' b'2' ... b'9'
Attributes: (12/70)
    node:                               RID27
    comment:                            Data produced by the OOI M2M API and ...
    publisher_email:                    
    sourceUrl:                          http://oceanobservatories.org/
    collection_method:                  recovered_inst
    stream:                             plims_a_adc_instrument
    ...                                 ...
    geospatial_lon_resolution:          0.1
    geospatial_vertical_units:          meters
    geospatial_vertical_resolution:     0.1
    geospatial_vertical_positive:       down
    lat:                                35.94988
    lon:                                -75.11943

In [32]:
# Check format of sample_filename parameter
adc_gc["sample_filename"].head().compute()

<xarray.DataArray 'sample_filename' (time: 5, string24: 5)> Size: 25B
array([[b'D', b'2', b'0', b'2', b'4'],
       [b'D', b'2', b'0', b'2', b'4'],
       [b'D', b'2', b'0', b'2', b'4'],
       [b'D', b'2', b'0', b'2', b'4'],
       [b'D', b'2', b'0', b'2', b'4']], dtype='|S1')
Coordinates:
  * time     (time) datetime64[ns] 40B 2024-04-03T14:16:10.610913792 ... 2024...
Dimensions without coordinates: string24
Attributes:
    _FillValue:   b'e'
    comment:      File name without extension as a globally unique identifier...
    coordinates:  time lat lon
    long_name:    Sample File Name

In [33]:
# Check whether dtype conversion shows correct strings
adc_gc["sample_filename"] = adc_gc.sample_filename.astype(str)
adc_gc["sample_filename"].head().compute()

<xarray.DataArray 'sample_filename' (time: 5, string24: 5)> Size: 100B
array([['D', '2', '0', '2', '4'],
       ['D', '2', '0', '2', '4'],
       ['D', '2', '0', '2', '4'],
       ['D', '2', '0', '2', '4'],
       ['D', '2', '0', '2', '4']], dtype='<U1')
Coordinates:
  * time     (time) datetime64[ns] 40B 2024-04-03T14:16:10.610913792 ... 2024...
Dimensions without coordinates: string24
Attributes:
    _FillValue:   b'e'
    comment:      File name without extension as a globally unique identifier...
    coordinates:  time lat lon
    long_name:    Sample File Name

In [34]:
adc_gc["sample_filename"]

<xarray.DataArray 'sample_filename' (time: 3186577, string24: 24)> Size: 306MB
array([['D', '2', '0', ..., '1', '9', '9'],
       ['D', '2', '0', ..., '1', '9', '9'],
       ['D', '2', '0', ..., '1', '9', '9'],
       ...,
       ['D', '2', '0', ..., '1', '9', '9'],
       ['D', '2', '0', ..., '1', '9', '9'],
       ['D', '2', '0', ..., '1', '9', '9']],
      shape=(3186577, 24), dtype='<U1')
Coordinates:
  * time     (time) datetime64[ns] 25MB 2024-04-03T14:16:10.610913792 ... 202...
Dimensions without coordinates: string24
Attributes:
    _FillValue:   b'e'
    comment:      File name without extension as a globally unique identifier...
    coordinates:  time lat lon
    long_name:    Sample File Name

In [17]:
# Save NetCDF locally to share file
adc_gc.to_netcdf(f"./plims_data/gc-recoveredinst-adc-{datetime.today()}.nc")

PermissionError: [Errno 13] Permission denied: 'c:\\Users\\kylene.cooley\\Documents\\GitHub\\pioneer-21\\plims_data\\gc-recoveredinst-adc-2025-10-13 11:22:40.328142.nc'

In [22]:
# Paste link directly from gold copy catalog into xr.load_dataset
adc_gc2 = xr.load_dataset("https://thredds.dataexplorer.oceanobservatories.org/thredds/dodsC/ooigoldcopy/public/CP10CNSM-RID27-07-PLIMSA000-recovered_inst-plims_a_adc_instrument/deployment0001_CP10CNSM-RID27-07-PLIMSA000-recovered_inst-plims_a_adc_instrument_20240403T141610.610914-20250405T004401.902189.nc")

In [23]:
adc_gc2

<xarray.Dataset> Size: 958MB
Dimensions:                     (obs: 3279801)
Coordinates:
  * obs                         (obs) int32 13MB 0 1 2 ... 3279799 3279800
    time                        (obs) datetime64[ns] 26MB 2024-04-03T14:16:10...
Data variables: (12/26)
    driver_timestamp            (obs) datetime64[ns] 26MB 2025-10-01T14:59:54...
    id                          (obs) |S64 210MB b'5c63ccf3-d26f-48c0-b329-71...
    inhibit_time_tr             (obs) float32 13MB 0.09436 0.09436 ... 101.5
    peak_b                      (obs) float32 13MB 0.0403 0.0403 ... 0.06607
    peak_a                      (obs) float32 13MB 3.467 3.467 ... 0.07058
    grab_time_start             (obs) float32 13MB 1.611 1.611 ... 1.201e+03
    ...                          ...
    run_time_tr                 (obs) float32 13MB 1.633 1.633 ... 1.201e+03
    roi_height                  (obs) float64 26MB 154.0 42.0 ... 106.0 138.0
    port_timestamp              (obs) datetime64[ns] 26MB 1900-01-01 ... 1900...
    deployment                  (obs) int32 13MB 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1
    preferred_timestamp         (obs) object 26MB b'internal_timestamp' ... b...
    sample_filename             (obs) object 26MB b'D20240403T141609_IFCB199'...
Attributes: (12/73)
    node:                               RID27
    comment:                            
    publisher_email:                    
    sourceUrl:                          http://oceanobservatories.org/
    collection_method:                  recovered_inst
    stream:                             plims_a_adc_instrument
    ...                                 ...
    geospatial_vertical_positive:       down
    lat:                                35.94988
    lon:                                -75.11943
    DODS.strlen:                        24
    DODS.dimName:                       string24
    DODS_EXTRA.Unlimited_Dimension:     obs

In [24]:
# Check whether new parameters are available
for var in adc_gc2.variables:
    if "sample" in var: print(var)

sample_timestamp
sample_adc_file_row_number
sample_filename


In [25]:
# When available, check the content of the new parameters
adc_gc2.sample_adc_file_row_number.head(20).compute()

<xarray.DataArray 'sample_adc_file_row_number' (obs: 20)> Size: 160B
array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20.])
Coordinates:
  * obs      (obs) int32 80B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
    time     (obs) datetime64[ns] 160B 2024-04-03T14:16:10.610913792 ... 2024...
Attributes:
    long_name:    Sample ADC File Row Number
    precision:    0
    comment:      Row number in adc file is ROI number; ROIs have nonzero roi...
    _ChunkSizes:  10000

In [ ]:
# Check edge cases - 7 ROIs from same trigger
# Downloaded data must include date 2024-09-10
samplefile = (
        adc_gc2.sample_filename==b'D20240910T042446_IFCB199'
    ).values.astype(bool)
sample_rows = adc_gc2.sample_adc_file_row_number[samplefile]
sample_triggers = adc_gc2.trigger[samplefile]
# Check for places where the trigger number for a ROI is the same as the previous one
trig_diff = np.where(np.diff(sample_triggers) == 0)
print(sample_rows.isel(obs=trig_diff[0]))
print(sample_triggers.isel(obs=trig_diff[0]))


<xarray.DataArray 'sample_adc_file_row_number' (obs: 7)> Size: 56B
array([188., 225., 226., 227., 228., 229., 230.])
Coordinates:
  * obs      (obs) int32 28B 1510057 1510094 1510095 ... 1510097 1510098 1510099
    time     (obs) datetime64[ns] 56B 2024-09-10T04:40:49.972002816 ... 2024-...
Attributes:
    long_name:    Sample ADC File Row Number
    precision:    0
    comment:      Row number in adc file is ROI number; ROIs have nonzero roi...
    _ChunkSizes:  10000
<xarray.DataArray 'trigger' (obs: 7)> Size: 56B
array([188., 224., 225., 225., 225., 225., 225.])
Coordinates:
  * obs      (obs) int32 28B 1510057 1510094 1510095 ... 1510097 1510098 1510099
    time     (obs) datetime64[ns] 56B 2024-09-10T04:40:49.972002816 ... 2024-...
Attributes:
    comment:                   Trigger number of the acquired data in sequence
    precision:                 0
    long_name:                 Trigger Number
    alternate_parameter_name:  trigger_number
    units:                     counts

: 

## Check data accuracy

Early versions of the PLIMS NetCDFs showed inaccuracies due to a loss of precision in the float data type (32-bit) for multiple parameters. These were changed to the double data type (64-bit), but the efficacy of the updates need to be confirmed.

In [ ]:
# Define routine to return any offset between the timestamp of record and sample start time
def test_time_delta(data):
    t_delta = data["internal_timestamp"] - data["sample_timestamp"]
    print(t_delta.head())
    check = (t_delta == 0)
    if check.all() == True:
        t_delta = "No offset."
    return t_delta

In [ ]:
# Test for a time delta in both HDR data streams
# Telemetered
tdel_hdrsci = test_time_delta(data_hdrsci)
print(tdel_hdrsci)
tdel_hdreng = test_time_delta(data_hdreng)
print(tdel_hdreng)
# Recovered Host
tdel_hdrsci2 = test_time_delta(data_hdrsci2)
print(tdel_hdrsci2)
tdel_hdreng2 = test_time_delta(data_hdreng2)
print(tdel_hdreng2)

## HDR Science data visualizations

## ADC Science data visualizations

In [ ]:
# Peak PMT A/B sample plot
fig, axs = plt.subplots(2, 1, figsize=(8,6), layout="tight")
# Peak PMTs during sampling on 1 May 2025
data_adcsci.peak_a.plot.scatter(x="time", s=20, edgecolors="none", ax=axs[0], label="Peak PMT A")
data_adcsci.peak_b.plot.scatter(x="time", s=20, edgecolors="none", ax=axs[0], label="Peak PMT B")
axs[0].set_ylabel("Peak PMT Output [V]")
axs[0].legend()
axs[0].grid()
# Peak PMTs during last sample on 1 May 2025
data_adcsci.peak_a[-2300:].plot.scatter(x="time", s=20, edgecolors="none", ax=axs[1], label="Peak PMT A")
data_adcsci.peak_b[-2300:].plot.scatter(x="time", s=20, edgecolors="none", ax=axs[1], label="Peak PMT B")
axs[1].set_ylabel("Peak PMT Output [V]")
axs[1].legend()
axs[1].grid()
fig.suptitle("Pioneer MAB Central Surface Mooring \nPlankton Imaging Sensor Peak PMT in May 2024")

In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(data_adcsci["grab_time_start"][-2300:], data_adcsci["roi_y"][-2300:], edgecolors="none")
plt.grid()
plt.ylabel("ROI$_y$")
plt.xlabel("Time of Flight [s]")

In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(data_adcsci["time"], data_adcsci["roi_y"], edgecolors="none")
plt.grid()
plt.ylabel("ROI$_y$")
plt.xlabel("Time")

In [ ]:
# Plot ROI width v. ROI height for 1 week of data in May 2024
plt.figure(figsize=(8,4))
plt.scatter(data_adcsci["roi_width"], data_adcsci["roi_height"], edgecolors="none", s=12)
plt.grid()
plt.ylabel("ROI height [pixels]")
plt.xlabel("ROI width [pixels]")
plt.title(f"ROI sizes from {np.datetime_as_string(
    data_adcsci.time[0].values, unit='D')} to {np.datetime_as_string(
    data_adcsci.time[-1].values, unit='D')}")

In [ ]:
# ROI width v. ROI height for 2 weeks of data in May 2024
plt.figure(figsize=(8,4))
plt.scatter(data_adcsci2["roi_width"], data_adcsci2["roi_height"], edgecolors="none", s=12)
plt.grid()
plt.ylabel("ROI height [pixels]")
plt.xlabel("ROI width [pixels]")
plt.title(f"ROI sizes from {np.datetime_as_string(
    data_adcsci2.time[0].values, unit='D')} to {np.datetime_as_string(
    data_adcsci2.time[-1].values, unit='D')}")

## Plots that combine HDR and ADC science data

In [ ]:
# Find peak PMT B values greater than 3 standard deviations from the mean
peak_b = data_adcsci2["peak_b"].compute()
mu = peak_b.mean()
sig = peak_b.std()
peak_b_3sig = peak_b.where(peak_b>(mu+3*sig), drop=True)

from xarray.groupers import TimeResampler
sample_mean_peak_b_3sig = peak_b_3sig.groupby(time=TimeResampler("h")).mean().dropna("time")

What if I want a sample with a high density of ROIs and the ROIs have strong chlorophyll signals on average?

In [ ]:
# Plot ROIS per ml and peak PMTB on the same axes
fig, ax = plt.subplots(figsize=(8,5), layout="constrained")
ax2 = plt.twinx(ax=ax)
pb = sample_mean_peak_b_3sig.plot.scatter(edgecolors="none", ax=ax, s=16, label="Sample mean Peak PMT B")
# sample_mean_peak_b_3sig.plot.line(x="time", ax=ax, label="Sample mean Peak PMT B")
data_hdrsci["rois_per_ml"].sel(time=slice("2024-05-14")).plot.scatter(c="black", marker="*", edgecolors="black", ax=ax2, s=60, label="Sample ROIs per ml")
ax.grid()
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9), framealpha=1)
plt.title("ROIs per ml and Sample Mean of Peak PMTB$>\mu_B+3 \sigma_B$")
plt.show()

In [ ]:
triggernum = data_adcsci2.trigger.resample(time="1s").asfreq()
triggernum.sel(time=slice("2024-05-03")).plot()
data_hdrsci["roi_count"].sel(time=slice("2024-05-03")).plot.scatter(c="black", marker="*", edgecolors="black", s=60, label="Sample ROIs per ml")

## Check new ADC instrument stream parameters

<ul>
    <li>sample_filename</li>
    <li>sample_adc_file_row_number</li>
</ul>

In [ ]:
# # Uncomment and run this cell if no data is saved locally
# # Setup parameters needed to request data
# refdes = "CP10CNSM-RID27-07-PLIMSA000"
# method = "recovered_inst"
# stream = "plims_a_adc_instrument"

# # Import vocabulary
# site, node, sensor = refdes.split('-', 2)
# vocab = get_vocabulary(site, node, sensor)
# # Use the gold copy THREDDs datasets
# thredds_url = M2M.get_thredds_url(refdes, method,
#                                   stream, beginDT="2024-06-10T00:00:00.0",
#                                   endDT="2024-06-11T00:00:00.0")
# # Get the THREDDs catalog
# thredds_catalog = M2M.get_thredds_catalog(thredds_url)
# thredds_catalog = [x for x in thredds_catalog if "blank" not in x]

# # Clean the THREDDs catalog
# sensor_files, ancillary_files = M2M.clean_catalog(thredds_catalog,
#                                                   stream)

# # Generate the urls to access and load the data
# sensor_files = [
#     re.sub("catalog.html\?dataset=",
#            M2M.URLS["dodsC"], file) for file in sensor_files
#     ]
# # Load the data
# with ProgressBar():
#     data_adcinst = xr.open_mfdataset(sensor_files, parallel=True,
#                                     preprocess=preprocess)

In [ ]:
# # Load data saved locally
# data_adcinst = xr.load_dataset(
#     "./plims_data/dev0_recovered-inst_adc-instrument_20240610T002501.nc")

In [ ]:
# # Check dataset content
# print(data_adcinst.source)
# data_adcinst

In [33]:
# Check whether new parameters are available
for var in data_adcsci3.variables:
    if "sample" in var: print(var)

sample_timestamp
sample_adc_file_row_number
sample_filename


In [34]:
# When available, check the content of the new parameters
data_adcsci3.sample_adc_file_row_number.head(20).compute()

<xarray.DataArray 'sample_adc_file_row_number' (time: 20)> Size: 160B
array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20.])
Coordinates:
  * time     (time) datetime64[ns] 160B 2024-09-10T00:24:57.241979392 ... 202...
Attributes:
    comment:      Row number in adc file is ROI number; ROIs have nonzero roi...
    precision:    0
    long_name:    Sample ADC File Row Number
    _ChunkSizes:  10000

In [ ]:
# Check edge cases - 2 ROIs from same trigger
# Downloaded data must include date 2024-06-10
# triggers = (
#       (data_hdreng.trigger>445)&(data_hdreng.trigger<455)
#   ).values.astype(bool)
samplefile = (
        data_adcsci3.sample_filename==b'D20240610T002501_IFCB199'
    ).values.astype(bool)
doubleroi = data_adcsci3.sample_adc_file_row_number[samplefile]
# print(doubleroi.sample_timestamp.head().compute())
# print(doubleroi.trigger.head(20).compute().values)
# print(doubleroi.sample_adc_file_row_number.head(20).compute().values)
doubleroi.isel(time=slice(445, 455)).compute()
# doubleroi.isel(time=slice(445, 455)).to_netcdf(
#     "./plims_data/dev0_recovered-inst_adc-instrument_20240610T002501.nc")

In [ ]:
doubleroi.isel(time=slice(455, 459)).compute()

In [ ]:
doubletrigger = data_adcsci3.trigger[samplefile]
doubletrigger.isel(time=slice(455, 459)).compute()

In [ ]:
# check last trigger and row number of sample
# line ending in ADC file added blank line to file, should be skipped
print(
    doubleroi.tail().compute(),
    doubletrigger.tail().compute()
)
print(data_adcsci3.trigger.isel(time=slice(1460, 1470)).compute())

In [ ]:
# check whether trigger number 88 is in the dataset
print(data_adcsci3.trigger.isel(time=slice(85, 93)).compute())

In [ ]:
data_adcsci3.isel(time=87).compute()

In [40]:
# Check for places where the file row number for a ROI is smaller than the previous one
row_diff = np.where(np.diff(data_adcsci3['sample_adc_file_row_number']) < 0)
row_diff

(array([  301,   544,  1016,  2002,  2583,  2855,  3129,  3465,  3897,
         4337,  5482,  6059,  6439,  7188,  7911,  8599,  9328,  9905,
        11221, 12354, 13848, 15298, 16385, 17021, 17609, 18225, 19423,
        19891, 20182, 20569, 20833, 21073, 21252, 21884, 22167, 22565,
        22927, 23397, 23739, 24122, 24838, 25964, 26456, 26864, 27305,
        27777, 28505, 28929, 29681, 30856, 34053, 37175, 39812, 42091,
        42736]),)

The triggers with a row number followed by a lower row number might be correct since I expect there to be multiple sample runs in the timeframe of data requested. Next we check the row number and trigger numbers for triggers around those indexes. 

In [43]:
for x in row_diff[0]:
    print(
        list(
            zip(data_adcsci3["sample_adc_file_row_number"][int(x-2):int(x+2)].values,
                data_adcsci3["trigger"][int(x-2):int(x+2)].values)
        )
    )

[(np.float64(300.0), np.float64(299.0)), (np.float64(301.0), np.float64(300.0)), (np.float64(302.0), np.float64(301.0)), (np.float64(1.0), np.float64(1.0))]
[(np.float64(241.0), np.float64(234.0)), (np.float64(242.0), np.float64(235.0)), (np.float64(243.0), np.float64(236.0)), (np.float64(1.0), np.float64(1.0))]
[(np.float64(470.0), np.float64(467.0)), (np.float64(471.0), np.float64(468.0)), (np.float64(472.0), np.float64(469.0)), (np.float64(1.0), np.float64(1.0))]
[(np.float64(984.0), np.float64(982.0)), (np.float64(985.0), np.float64(983.0)), (np.float64(986.0), np.float64(984.0)), (np.float64(1.0), np.float64(1.0))]
[(np.float64(579.0), np.float64(575.0)), (np.float64(580.0), np.float64(576.0)), (np.float64(581.0), np.float64(577.0)), (np.float64(1.0), np.float64(1.0))]
[(np.float64(270.0), np.float64(269.0)), (np.float64(271.0), np.float64(270.0)), (np.float64(272.0), np.float64(271.0)), (np.float64(1.0), np.float64(1.0))]
[(np.float64(272.0), np.float64(272.0)), (np.float64(273.0

In [44]:
# Check for places where the trigger number for a ROI is the same as the previous one
trig_diff = np.where(np.diff(data_adcsci3['trigger']) == 0)
trig_diff[0]

array([   99,   490,   527,   528,   529,   530,   531,   532,   845,
         901,   927,  1419,  1839,  2072,  2086,  2164,  2405,  2759,
        3916,  4408,  5065,  5789,  6038,  7070,  7151,  7938,  8021,
        8300,  8433,  8813,  8814,  9608,  9790, 10220, 10625, 10676,
       11235, 11893, 11981, 11993, 12751, 13634, 15514, 17110, 18534,
       18810, 19289, 19463, 20139, 20154, 20528, 20775, 21290, 21291,
       21422, 21909, 21917, 22110, 22356, 22576, 22583, 22914, 22931,
       23398, 23538, 23682, 24529, 24709, 24910, 25426, 25468, 25730,
       26287, 26308, 26542, 26563, 26565, 26566, 26567, 26600, 27252,
       27384, 27675, 27684, 27881, 28109, 28237, 28573, 29395, 29429,
       29430, 29795, 30177, 30495, 30887, 30927, 31487, 31608, 31755,
       32004, 32447, 32528, 32579, 33242, 33337, 33787, 33898, 33967,
       33993, 34216, 34324, 34885, 35106, 35255, 35672, 35844, 35929,
       36289, 36403, 36405, 36606, 36656, 36851, 37486, 37564, 37900,
       38017, 38054,

In [58]:
results = dict()
for x in trig_diff[0]:
    results[str(x)] = list(
        zip(data_adcsci3["sample_adc_file_row_number"][int(x-1):int(x+2)].values.astype(int),
            data_adcsci3["trigger"][int(x-1):int(x+2)].values.astype(int))
    )
results

{'99': [(np.int64(99), np.int64(99)),
  (np.int64(100), np.int64(100)),
  (np.int64(101), np.int64(100))],
 '490': [(np.int64(188), np.int64(188)),
  (np.int64(189), np.int64(189)),
  (np.int64(190), np.int64(189))],
 '527': [(np.int64(225), np.int64(224)),
  (np.int64(226), np.int64(225)),
  (np.int64(227), np.int64(225))],
 '528': [(np.int64(226), np.int64(225)),
  (np.int64(227), np.int64(225)),
  (np.int64(228), np.int64(225))],
 '529': [(np.int64(227), np.int64(225)),
  (np.int64(228), np.int64(225)),
  (np.int64(229), np.int64(225))],
 '530': [(np.int64(228), np.int64(225)),
  (np.int64(229), np.int64(225)),
  (np.int64(230), np.int64(225))],
 '531': [(np.int64(229), np.int64(225)),
  (np.int64(230), np.int64(225)),
  (np.int64(231), np.int64(225))],
 '532': [(np.int64(230), np.int64(225)),
  (np.int64(231), np.int64(225)),
  (np.int64(232), np.int64(225))],
 '845': [(np.int64(300), np.int64(300)),
  (np.int64(301), np.int64(301)),
  (np.int64(302), np.int64(301))],
 '901': [(np.

In [46]:
print(data_adcsci3["sample_filename"][528].values)

np.bytes_(b'D20240910T042446_IFCB199')
